In [ ]:
#conda install geopandas

In [ ]:
import pandas as pd
import quality_func

import geopandas as gpd

In [ ]:
zone = quality_func.load_zone_data()
zone.shape

In [ ]:
zone.head(5)

In [ ]:
files = ["data/uber-trip-data/uber-raw-data-apr14.csv", "data/uber-trip-data/uber-raw-data-may14.csv", "data/uber-trip-data/uber-raw-data-jun14.csv", 'data/uber-trip-data/uber-raw-data-jul14.csv', 'data/uber-trip-data/uber-raw-data-aug14.csv', 'data/uber-trip-data/uber-raw-data-sep14.csv']

raw_2014 = quality_func.load_2014_data(files)
raw_2014.shape

In [ ]:
raw_2014.head(5)

In [ ]:
raw_2015 = quality_func.load_2015_data('data/uber-trip-data/uber-raw-data-janjune-15.csv')

# Explore dataset

In [ ]:
dataframes_viz = [zone, raw_2014, raw_2015]
[print(f"Display of dataset: {display(df.head())}") for df in dataframes_viz]

In [ ]:
zones_official = gpd.read_file("data/taxi_zones/taxi_zones/taxi_zones.shp")[["LocationID", "geometry"]].to_crs("EPSG:4326")

In [ ]:
loc_2014 = gpd.GeoDataFrame(
    raw_2014,
    geometry=gpd.points_from_xy(raw_2014["Lon"], raw_2014["Lat"]),
    crs="EPSG:4326"
)

In [ ]:
dataset_2014 = gpd.sjoin(loc_2014, zones_official, how="left", predicate="within").drop(columns=["index_right"])

In [ ]:
dataset_2014.head(5)

In [ ]:
zones_official["rep_pt"] = zones_official.geometry.representative_point()
zones_official["lon"] = zones_official["rep_pt"].x
zones_official["lat"] = zones_official["rep_pt"].y

In [ ]:
raw_2015 = raw_2015.rename(columns={"locationID": "LocationID"})

In [ ]:
dataset_2015 = raw_2015.merge(zones_official[["LocationID", "lat", "lon"]], on="LocationID", how="left")

In [ ]:
dataset_2015.head()

In [ ]:
display(zone.head(1),zones_official.head(1) ,dataset_2014.head(1), dataset_2015.head(1))

In [ ]:
dataset_zone = pd.merge(zone, zones_official[["LocationID", "lat", "lon"]], on="LocationID", how="left")

In [ ]:
dataset_2014 = dataset_2014.drop(columns=["geometry"])

In [ ]:
display(dataset_zone.head(1), dataset_2014.head(1), dataset_2015.head(1))

In [ ]:
dataset_2014 = dataset_2014.rename(columns={"Date/Time": "Date", "Lat": "lat", "Lon": "lon"})
dataset_2015 = dataset_2015.rename(columns={"Pickup_date": "Date", "Dispatching_base_num": "Base"})


In [ ]:
dataset = pd.merge(dataset_2014, dataset_2015, how="left", on=["LocationID"])

In [ ]:
dataset.head()